# 04 — Preparação raster LULC

Rasterização das camadas vetoriais LULC harmonizadas através do campo `id_harm`.

O raster de referência define o CRS, a resolução, a extensão e o alinhamento de todos os outputs. O valor `0` é utilizado como NoData.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import rasterio as rio

sys.path.append("/code/scripts")

from glass.dtt.rst.torst import shp_to_rst
from raster_utils import check_raster_alignment, raster_ids

## 1. Seleção dos produtos

Alterar apenas a variável `source`.

- `cos`: COS 1995, 2007, 2010, 2015 e 2018 — Região Centro;
- `siose`: SIOSE 2005, 2009, 2011 e 2014 — Extremadura;
- `siose_ar`: SIOSE AR 2017 e 2020 — Extremadura.

In [ ]:
# ALTERAR APENAS ESTA VARIÁVEL

source = "siose_ar"

valid_sources = ["cos", "siose", "siose_ar"]

if source not in valid_sources:
    raise ValueError(
        f"Fonte inválida: {source}. "
        f"Opções: {valid_sources}"
    )

area = "centro" if source == "cos" else "extremadura"

print("Área:", area)
print("Fonte:", source)

## 2. Caminhos

In [ ]:
base = f"/code/data/processed/{area}"

in_dir = f"{base}/lulc/harmonized_vectors"
out_dir = f"{base}/lulc/rasters"

ref = (
    f"{base}/topo/derived/"
    f"dem_{area}_clip.tif"
)

Path(out_dir).mkdir(
    parents=True,
    exist_ok=True
)

if not Path(ref).exists():
    raise FileNotFoundError(
        f"Raster de referência não encontrado: {ref}"
    )

print("Raster de referência:", ref)
print("Pasta de entrada:", in_dir)
print("Pasta de saída:", out_dir)

## 3. Produtos a rasterizar

In [ ]:
products_by_source = {
    "cos": [
        "cos_1995",
        "cos_2007",
        "cos_2010",
        "cos_2015",
        "cos_2018",
    ],
    "siose": [
        "siose_2005",
        "siose_2009",
        "siose_2011",
        "siose_2014",
    ],
    "siose_ar": [
        "siose_ar_2017",
        "siose_ar_2020",
    ],
}

products = products_by_source[source]
lulc_files = {}

for product in products:
    vector = f"{in_dir}/{product}_harm.gpkg"
    layer = f"{product}_harm"

    if source == "cos":
        year = product.split("_")[-1]
        raster = f"{out_dir}/lulc_{year}.tif"
    else:
        raster = f"{out_dir}/lulc_{product}.tif"

    lulc_files[product] = {
        "file": vector,
        "layer": layer,
        "out": raster,
    }

missing = [
    cfg["file"]
    for cfg in lulc_files.values()
    if not Path(cfg["file"]).exists()
]

if missing:
    raise FileNotFoundError(
        "Ficheiros harmonizados não encontrados:\n"
        + "\n".join(missing)
    )

for product, cfg in lulc_files.items():
    print(product)
    print("  Entrada:", cfg["file"])
    print("  Saída:", cfg["out"])

## 4. Rasterização

In [ ]:
from pathlib import Path

import numpy as np
import pyogrio
import rasterio as rio

from rasterio.crs import CRS
from rasterio.features import rasterize
from rasterio.windows import Window, bounds


def rasterize_lulc_blocks(
    vector,
    layer,
    field,
    reference,
    output,
    block_size=512,
):
    """
    Rasteriza uma layer vetorial por blocos, utilizando
    a grelha do raster de referência.

    Esta abordagem evita carregar todas as geometrias
    e o raster completo para memória.
    """

    vector_info = pyogrio.read_info(
        vector,
        layer=layer,
    )

    vector_crs = CRS.from_user_input(
        vector_info["crs"]
    )

    if field not in vector_info["fields"]:
        raise ValueError(
            f"O campo '{field}' não existe em {vector}."
        )

    output = Path(output)

    for suffix in [
        "",
        ".aux.xml",
        ".ovr",
        ".msk",
    ]:
        Path(str(output) + suffix).unlink(
            missing_ok=True
        )

    with rio.open(reference) as ref_src:

        if vector_crs != ref_src.crs:
            raise ValueError(
                "O CRS do vetor não coincide com o "
                "raster de referência.\n"
                f"Vetor: {vector_crs}\n"
                f"Raster: {ref_src.crs}"
            )

        profile = ref_src.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="uint16",
            nodata=0,
            compress="lzw",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="IF_SAFER",
            SPARSE_OK="TRUE",
        )

        total_rows = int(
            np.ceil(
                ref_src.height / block_size
            )
        )

        total_cols = int(
            np.ceil(
                ref_src.width / block_size
            )
        )

        total_blocks = (
            total_rows
            * total_cols
        )

        processed = 0
        features_read = 0

        with rio.open(
            output,
            "w",
            **profile,
        ) as dst:

            for row_off in range(
                0,
                ref_src.height,
                block_size,
            ):
                for col_off in range(
                    0,
                    ref_src.width,
                    block_size,
                ):
                    height = min(
                        block_size,
                        ref_src.height - row_off,
                    )

                    width = min(
                        block_size,
                        ref_src.width - col_off,
                    )

                    window = Window(
                        col_off=col_off,
                        row_off=row_off,
                        width=width,
                        height=height,
                    )

                    window_bounds = bounds(
                        window,
                        ref_src.transform,
                    )

                    features = (
                        pyogrio.read_dataframe(
                            vector,
                            layer=layer,
                            columns=[field],
                            where=f"{field} IS NOT NULL",
                            bbox=window_bounds,
                            use_arrow=True,
                        )
                    )

                    if not features.empty:
                        features = features.loc[
                            features.geometry.notna()
                            & ~features.geometry.is_empty
                        ]

                    if not features.empty:
                        shapes = (
                            (
                                geometry,
                                int(value),
                            )
                            for geometry, value
                            in zip(
                                features.geometry,
                                features[field],
                            )
                        )

                        block = rasterize(
                            shapes=shapes,
                            out_shape=(
                                int(height),
                                int(width),
                            ),
                            transform=(
                                ref_src.window_transform(
                                    window
                                )
                            ),
                            fill=0,
                            dtype="uint16",
                            all_touched=False,
                        )

                        dst.write(
                            block,
                            1,
                            window=window,
                        )

                        features_read += len(
                            features
                        )

                    processed += 1

                    if (
                        processed % 25 == 0
                        or processed == total_blocks
                    ):
                        print(
                            f"Blocos: {processed}/"
                            f"{total_blocks} "
                            f"({processed / total_blocks:.1%})"
                        )

        print("\nRaster criado:", output)
        print(
            "Referências a feições processadas:",
            f"{features_read:,}",
        )

In [ ]:
for product, cfg in lulc_files.items():

    print(
        f"\n{'=' * 60}\n"
        f"Rasterizar {product}\n"
        f"{'=' * 60}"
    )

    rasterize_lulc_blocks(
        vector=cfg["file"],
        layer=cfg["layer"],
        field="id_harm",
        reference=ref,
        output=cfg["out"],
        block_size=512,
    )

    check_raster_alignment(
        raster=cfg["out"],
        template=ref,
    )

    print(
        f"{product} -> {cfg['out']}"
    )

In [ ]:
for product, cfg in lulc_files.items():
    output = Path(cfg["out"])
    output.unlink(missing_ok=True)

    shp_to_rst(
        shp=cfg["file"],
        inSource="id_harm",
        cellsize=None,
        nodata=0,
        outRaster=cfg["out"],
        rst_template=ref,
        lyrname=cfg["layer"],
        api="gdal",
        dtype="UInt16",
        rtype=int,
    )

    check_raster_alignment(
        raster=cfg["out"],
        template=ref
    )

    print(f"{product} -> {cfg['out']}")

## 5. Verificação dos resultados

In [ ]:
for product, cfg in lulc_files.items():
    ids = raster_ids(cfg["out"])

    with rio.open(cfg["out"]) as src:
        print(f"\nProduto: {product}")
        print("Raster:", cfg["out"])
        print("Número de classes:", len(ids))
        print("IDs presentes:", ids)
        print("CRS:", src.crs)
        print("Resolução:", src.res)
        print("Dimensão:", src.shape)
        print("NoData:", src.nodata)
        print("Tipo:", src.dtypes[0])

## 6. Resumo dos outputs

In [ ]:
print("Rasterização concluída")
print("-" * 40)
print("Área:", area)
print("Fonte:", source)
print("Raster de referência:", ref)

for product, cfg in lulc_files.items():
    print(f"{product}: {cfg['out']}")